In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

# Filter out rows where prognostic subtype is the string 'NA'
filtered_metadata_df = metadata_df[metadata_df['prognostic subtype'] != 'NA']

print(filtered_metadata_df.shape)
filtered_metadata_df


(46, 7)


,sample_title,tissue,gender,age,survival time,survival,prognostic subtype
samples,,,,,,,
GSM4998233,2004-950RNAseq,Brain Metastatic Tissue,Male,67,35.8,1,Poor
GSM4998234,2006-262-RNAseq,Brain Metastatic Tissue,Male,55,1,0,Good
GSM4998235,2006-76RNASeq,Brain Metastatic Tissue,Male,61,1.1,1,Good
GSM4998236,2007-109RNASeq,Brain Metastatic Tissue,Female,61,3.2,1,Good
GSM4998237,2007-263-RNAseq,Brain Metastatic Tissue,Female,61,2.6,1,Good
GSM4998238,2007-45RNAseq,Brain Metastatic Tissue,Female,45,8.6,1,Cluster 1
GSM4998239,2007-550RNAseq,Brain Metastatic Tissue,Female,81,58,0,Poor
GSM4998240,2007-563RNAseq,Brain Metastatic Tissue,Female,47,113.2,0,Poor
GSM4998241,2007-601RNAseq,Brain Metastatic Tissue,Female,47,10,1,Others


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_98340/3050319327.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,gender,age,survival time,survival,prognostic subtype,DDX11L1,WASH7P,MIR6859-1,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM4998233,2004-950RNAseq,Brain Metastatic Tissue,Male,67,35.8,1,Poor,4,173,5,...,3013,20,21,18,1981,323,64,2783,44,536
GSM4998234,2006-262-RNAseq,Brain Metastatic Tissue,Male,55,1,0,Good,9,204,11,...,31743,29,33,28,20092,1861,657,18557,118,1158
GSM4998235,2006-76RNASeq,Brain Metastatic Tissue,Male,61,1.1,1,Good,3,337,6,...,6032,13,12,10,5681,650,337,6278,22,756
GSM4998236,2007-109RNASeq,Brain Metastatic Tissue,Female,61,3.2,1,Good,0,146,6,...,10508,41,40,31,5531,609,248,10292,95,1309
GSM4998237,2007-263-RNAseq,Brain Metastatic Tissue,Female,61,2.6,1,Good,14,355,16,...,56838,173,187,168,44605,2439,616,37793,256,832
GSM4998238,2007-45RNAseq,Brain Metastatic Tissue,Female,45,8.6,1,Cluster 1,6,82,0,...,2998,20,16,14,2298,274,90,2867,96,382
GSM4998239,2007-550RNAseq,Brain Metastatic Tissue,Female,81,58,0,Poor,7,294,10,...,6387,30,35,26,4762,581,161,6691,80,299
GSM4998240,2007-563RNAseq,Brain Metastatic Tissue,Female,47,113.2,0,Poor,6,271,5,...,4696,14,12,11,3166,355,128,3470,74,597
GSM4998241,2007-601RNAseq,Brain Metastatic Tissue,Female,47,10,1,Others,4,187,3,...,3056,19,23,14,2239,351,129,3436,59,437


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
